# Classification Probability Layers

Ensemble posterior、calibration、reliability、conformalの責務をprobability-spaceで整理します。


## このNotebookで学ぶこと

39番でクラス確率を出せるようになっても、その数字が信頼できるとは限りません。ここでは **予測確率の分散（ensemble）**、**確率の校正（calibration）**、**予測の信頼性（reliability）** の役割を分けて理解します。Conformalは概念だけを説明し、このNotebookでは予測集合を計算しません。

In [ ]:
import torch

from robotorchan.models.classification import (
    ClassificationEnsemblePosterior,
    TemperatureScalingCalibrator,
    classification_nll,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

### 今回の合成データ

`train_X` と `train_Y` は二値分類の位置とラベルです。`grid` は図を描くための51点です。ただし、次に作るensembleは**このデータで学習した分類器から作るわけではありません**。合成した3通りの確率曲線を比較します。

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Probability ensemble posterior

### 3つの確率曲線を合成する

`p`、`p**0.8`、`p**1.2` の3通りでクラス1確率を作っています。どのメンバーでも、クラス0確率とクラス1確率の合計は1です。

`posterior.mean` はメンバー平均、`posterior.variance` はメンバー間の確率のばらつきを表します。**観測ラベルの分散** と同一ではありません。

In [ ]:
p = grid.squeeze(-1)
members = torch.stack([
    torch.stack((1.0 - p, p), dim=-1),
    torch.stack((1.0 - p.pow(0.8), p.pow(0.8)), dim=-1),
    torch.stack((1.0 - p.pow(1.2), p.pow(1.2)), dim=-1),
])
posterior = ClassificationEnsemblePosterior(members)
print(posterior.mean.shape)
print(posterior.variance.shape)


## 3. Probability calibration

### Temperature scalingの読み方

`raw` は各行が `[P(y=0), P(y=1)]` の確率、`targets` は正解ラベルです。`calibrator.fit(raw, targets)` は温度パラメータを推定し、`calibrator(raw)` が補正後の確率を返します。

この4点は校正にも評価にも使われているため、表示されるNLLの改善は**汎化性能を証明しません**。実運用では校正用と最終評価用のデータを分けます。

In [ ]:
raw = torch.tensor([[0.90, 0.10], [0.65, 0.35], [0.30, 0.70], [0.05, 0.95]])
targets = torch.tensor([0, 0, 1, 1])
calibrator = TemperatureScalingCalibrator()
calibrator.fit(raw, targets)
calibrated = calibrator(raw)
print(classification_nll(raw, targets).item())
print(classification_nll(calibrated, targets).item())


## 4. Reliability / conformalとの違い

Reliability/OODはentropy・disagreement・input distanceを分離して診断します。Split conformalはheld-out calibration setからprediction setを構成します。probability calibrationとは別レイヤーです。

実運用ではmodel fitting、probability calibration、conformal calibrationに使うデータの役割を明示してください。

## 5. Probability-space候補選択とラベル評価

このNotebookのensembleは合成確率メンバーであり、学習済み分類器ではありません。Probability calibrationは候補最適化ではなく確率の補正です。ここではensemble disagreementによる候補選択を示します。


### 次の候補はどこか

`posterior.variance[..., 1]` はクラス1確率のメンバー間分散です。その最大点を `grid` から選びます。これは学習済みGPから獲得関数を最大化した結果ではなく、説明用の有限候補選択です。

In [ ]:
candidate_disagreement = posterior.variance[..., 1]
candidate = grid[candidate_disagreement.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate)
print("observed label:", observed_label)


## 6. 図で読む：予測平均、メンバー間分散、確率校正

左の図では3つの合成メンバーと平均予測を比較します。右の図はメンバー間の分散です。校正前後の4点も別図で比較します。これらは概念の可視化であり、モデルの実測精度評価ではありません。

In [ ]:
import matplotlib.pyplot as plt

xs = grid.squeeze(-1).detach().cpu().numpy()
member_values = members[..., 1].detach().cpu().numpy()
mean_values = posterior.mean[..., 1].detach().cpu().numpy()
variance_values = candidate_disagreement.detach().cpu().numpy()
selected_x = candidate.detach().cpu().reshape(-1)[0].item()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, values in enumerate(member_values):
    axes[0].plot(xs, values, alpha=0.65, label=f"Member {idx + 1}")
axes[0].plot(xs, mean_values, linewidth=2, label="Ensemble mean")
axes[0].set_ylabel("P(y=1)")
axes[0].set_xlabel("Input x")
axes[0].legend()
axes[1].plot(xs, variance_values, label="Member disagreement")
axes[1].axvline(selected_x, linestyle="--", color="gray", label="Candidate")
axes[1].set_ylabel("Variance of P(y=1)")
axes[1].set_xlabel("Input x")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
indices = torch.arange(raw.shape[0]).cpu().numpy()
ax.plot(indices, raw[:, 1].detach().cpu().numpy(), "o-", label="Before")
ax.plot(indices, calibrated[:, 1].detach().cpu().numpy(), "s--", label="After")
ax.scatter(indices, targets.detach().cpu().numpy(), marker="x", label="True label")
ax.set_xticks(indices)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Sample index")
ax.set_ylabel("P(y=1) / label")
ax.set_title("Temperature scaling on demonstration data")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 読み取りと評価の注意

- モデル間の不一致が大きい場所と、平均予測が0.5付近の場所は同一とは限りません。
- 校正後の確率が大きく変わっても、それだけで分類性能が改善したとはいえません。
- 実際の校正性能は、独立した評価データに対してNLL・Brier Score・reliability diagram等で確認します。

## 初学者向け補足：確率を出した後にも評価が必要

分類器が `0.9` と予測しても、実際に同じような予測を100件集めて約90件がクラス1になるとは限りません。確率の数値と実際の頻度が一致する性質を **calibration（確率校正）** と呼びます。

| 概念 | 答えたい問い |
|---|---|
| Ensemble | 複数モデルの予測はどの程度一致するか |
| Calibration | 予測確率と実際の頻度は整合しているか |
| Reliability / OOD | 未知の入力や信頼しづらい予測を検出できるか |
| Conformal prediction | 仮定のもとで所定の被覆率を目指す予測集合を作れるか |

### このNotebookで注意すること

このNotebookのensembleは **合成した確率メンバー** を用いる説明例です。学習済みGP分類器を複数訓練した性能実験ではありません。

校正を評価するときは、学習・校正・最終評価のデータを適切に分離します。Brier ScoreやLog Lossは確率の質を見る指標ですが、Accuracyとは測っているものが異なります。Conformalの被覆率も、必要な交換可能性などの仮定を満たすか確認してください。

### 読み取りの練習

予測確率が0.5に近い点と、メンバー間の意見が割れる点は必ずしも同じではありません。各セルの出力を見ながら、どの指標がどちらを強調するか考えてみてください。